In [47]:
# Import libraries

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

print("Libraries imported successfully!")

Libraries imported successfully!


In [48]:
# Load the raw data

data_folder = Path("../data/raw")

skills = pd.read_csv(data_folder / "skills_rows.csv")
vacancies = pd.read_csv(data_folder / "vacancies_rows.csv")
vacancy_skills = pd.read_csv(data_folder / "vacancy_skills_rows.csv")

print("Skills:", skills.shape)
print("Vacancies:", vacancies.shape)
print("Vacancy-Skills:", vacancy_skills.shape)

Skills: (2033, 3)
Vacancies: (2154, 10)
Vacancy-Skills: (13032, 2)


In [49]:
# Make copies 

skills_clean = skills.copy()
vacancies_clean = vacancies.copy()
vacancy_skills_clean = vacancy_skills.copy()

In [50]:
# Clean column names

skills_clean.columns = skills_clean.columns.str.strip().str.lower()

vacancies_clean.columns = vacancies_clean.columns.str.strip().str.lower()

vacancy_skills_clean.columns = (
    vacancy_skills_clean.columns
    .str.strip()
    .str.lower()
)

print(vacancies_clean.columns.tolist())

['id', 'created_at', 'title', 'company', 'source_url', 'description_raw', 'experience_years', 'published_at', 'location', 'experience_level']


In [51]:
# Clean text columns

text_columns = [
    "title",
    "company",
    "location",
    "experience_level"
]

for column in text_columns:
    vacancies_clean[column] = (
        vacancies_clean[column]
        .astype("string")
        .str.strip()
    )

In [52]:
vacancies_clean[["title", "company", "location", "experience_level"]].head()

,title,company,location,experience_level
0,Data Analyst,CRG,<NA>,<NA>
1,Data Analyst,Go Offer,<NA>,<NA>
2,Data Analyst (f/m/d),ITRex Group,<NA>,<NA>
3,Data Analyst - Fully Remote,Haystack,<NA>,<NA>
4,Data Analyst | Remote,Crossing Hurdles,<NA>,<NA>


In [53]:
# Clean the skills table

skills_clean["name"] = (
    skills_clean["name"]
    .astype("string")
    .str.strip()
)

In [54]:
# Convert dates

vacancies_clean["created_at"] = pd.to_datetime(
    vacancies_clean["created_at"],
    errors="coerce",
    utc=True
)

vacancies_clean["published_at"] = pd.to_datetime(
    vacancies_clean["published_at"],
    errors="coerce",
    utc=True
)

skills_clean["created_at"] = pd.to_datetime(
    skills_clean["created_at"],
    errors="coerce",
    utc=True
)

print(vacancies_clean.dtypes)

id                                int64
created_at          datetime64[ns, UTC]
title                    string[python]
company                  string[python]
source_url                       object
description_raw                  object
experience_years                float64
published_at        datetime64[ns, UTC]
location                 string[python]
experience_level         string[python]
dtype: object


In [55]:
# Analyze missing values 

missing_summary = pd.DataFrame({
    "Missing_Count": vacancies_clean.isnull().sum(),
    "Missing_Percentage": (
        vacancies_clean.isnull().mean() * 100
    ).round(2)
})

missing_summary.sort_values(
    "Missing_Percentage",
    ascending=False
)

,Missing_Count,Missing_Percentage
created_at,2120,98.42
experience_years,321,14.90
description_raw,208,9.66
location,25,1.16
published_at,16,0.74
experience_level,16,0.74
id,0,0.00
title,0,0.00
company,0,0.00
source_url,0,0.00


In [56]:
vacancies_clean["company"].value_counts().head(30)

company
Unknown                                    100
Upwork                                      26
Insight Global                              18
Dice                                        13
Meta                                        13
EDWARD JONES                                11
AARATECH                                    10
Turing.com                                   9
UnitedHealth Group                           8
Michael Page                                 8
Corporate                                    8
Crossing Hurdles                             7
Brooksource                                  7
Kforce Inc                                   7
Get It Recruit - Information Technology      6
Stripe                                       6
Microsoft                                    6
Emprego                                      6
S&P Global                                   6
Revolut                                      6
Talentify.io                                 5
CGI  

In [57]:
# Clean company names
 
vacancies_clean["company"] = (
    vacancies_clean["company"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [58]:
# Clean locations

vacancies_clean["location"] = (
    vacancies_clean["location"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [59]:
vacancies_clean["location"].value_counts().head(30)

location
Anywhere                                                     187
United States                                                154
New York, NY                                                  67
Kuala Lumpur                                                  47
Singapore                                                     32
San Francisco, CA                                             29
Austin, TX                                                    24
India                                                         24
Paris, France                                                 23
Los Angeles, CA                                               23
Chicago, IL                                                   22
United Kingdom                                                19
Washington, DC                                                19
Atlanta, GA                                                   18
Petaling                                                      17
Charlotte, NC   

In [60]:
# Investigate experience

print("Missing experience:",
      vacancies_clean["experience_years"].isna().sum())

print("\nZero experience:",
      (vacancies_clean["experience_years"] == 0).sum())

print("\nNegative experience:",
      (vacancies_clean["experience_years"] < 0).sum())

print("\nMaximum experience:",
      vacancies_clean["experience_years"].max())

Missing experience: 321

Zero experience: 1011

Negative experience: 0

Maximum experience: 25.0


In [61]:
vacancies_clean["title_normalized"] = (
    vacancies_clean["title"]
    .str.lower()
    .str.strip()
)

vacancies_clean[
    ["title", "title_normalized"]
].head(20)

,title,title_normalized
0,Data Analyst,data analyst
1,Data Analyst,data analyst
2,Data Analyst (f/m/d),data analyst (f/m/d)
3,Data Analyst - Fully Remote,data analyst - fully remote
4,Data Analyst | Remote,data analyst | remote
5,Data Analyst | Remote,data analyst | remote
6,Data Analyst (Product),data analyst (product)
7,Data Analyst,data analyst
8,Reporting Analyst | Remote,reporting analyst | remote
9,Data Analyst,data analyst


In [62]:
# Validate the vacancy-skill relationship

print("Total vacancies:",
      vacancies_clean["id"].nunique())

print("Vacancies with skills:",
      vacancy_skills_clean["vacancy_id"].nunique())

skill_coverage = (
    vacancy_skills_clean["vacancy_id"].nunique()
    / vacancies_clean["id"].nunique()
    * 100
)

print(f"Skill coverage: {skill_coverage:.2f}%")

Total vacancies: 2154
Vacancies with skills: 1873
Skill coverage: 86.95%


In [63]:
# Validate skill IDs

valid_skill_ids = set(skills_clean["id"])

invalid_skill_ids = (
    ~vacancy_skills_clean["skill_id"].isin(valid_skill_ids)
)

print(
    "Invalid skill IDs:",
    invalid_skill_ids.sum()
)

Invalid skill IDs: 0


In [64]:
# Validate vacancy IDs

valid_vacancy_ids = set(vacancies_clean["id"])

invalid_vacancy_ids = (
    ~vacancy_skills_clean["vacancy_id"].isin(valid_vacancy_ids)
)

print(
    "Invalid vacancy IDs:",
    invalid_vacancy_ids.sum()
)

Invalid vacancy IDs: 0


## Experience Analysis

Before handling missing experience values, we investigate
whether zero experience represents genuine entry-level jobs
or missing/unknown information.

In [65]:
# Experience level distribution

vacancies_clean["experience_level"].value_counts(dropna=False)

experience_level
Not specified       947
Middle              368
Entry level         271
Senior              229
Junior              119
Not Applicable       89
Mid-Senior level     53
Lead                 29
Associate            28
<NA>                 16
Internship            5
Name: count, dtype: Int64

In [66]:
# Experience vs experience level

pd.crosstab(
    vacancies_clean["experience_level"],
    vacancies_clean["experience_years"].eq(0)
)

experience_years,False,True
experience_level,,
Associate,27,1
Entry level,61,210
Internship,0,5
Junior,97,22
Lead,26,3
Mid-Senior level,47,6
Middle,328,40
Not Applicable,76,13
Not specified,264,683


In [67]:
# Look at zero-experience jobs

zero_experience = vacancies_clean[
    vacancies_clean["experience_years"] == 0
]

zero_experience[
    ["title", "company", "experience_years", "experience_level"]
].head(20)

,title,company,experience_years,experience_level
3,Data Analyst - Fully Remote,Haystack,0.0,<NA>
4,Data Analyst | Remote,Crossing Hurdles,0.0,<NA>
5,Data Analyst | Remote,Crossing Hurdles,0.0,<NA>
12,Data Analyst,Go Offer,0.0,<NA>
15,Junior Data Analyst,DetaPent,0.0,<NA>
16,Business Data Analyst,Athsai,0.0,Associate
40,Data Analyst,The Boothby Group,0.0,Entry level
44,Data Analyst,Total Recruitment Solutions,0.0,Entry level
54,"Clinical Data Analyst (FSP), South Africa",Parexel,0.0,Senior
71,Twitter Data Analyst - Code Tweets,Upwork,0.0,Entry level


In [68]:
# Look at missing-experience jobs

missing_experience = vacancies_clean[
    vacancies_clean["experience_years"].isna()
]

missing_experience[
    ["title", "company", "experience_years", "experience_level"]
].head(20)

,title,company,experience_years,experience_level
7,Data Analyst,Go Offer,NaN,<NA>
8,Reporting Analyst | Remote,Crossing Hurdles,NaN,<NA>
11,Data Analyst | Remote,Crossing Hurdles,NaN,<NA>
14,Reporting Analyst | Remote,Crossing Hurdles,NaN,<NA>
20,Data Analyst,SamSearch,NaN,Entry level
29,Data Analyst (Finance),Revolut,NaN,Not Applicable
48,DATA ANALYST,Nigel Frank International,NaN,Middle
53,Global Business Data Analyst,Dentons,NaN,Senior
69,Data Analyst,Upwork,NaN,Junior
98,Healthcare Data Analyst,Dice,NaN,Senior


In [69]:
vacancies_clean = vacancies_clean.drop(
    columns=["created_at"]
)

skills_clean = skills_clean.drop(
    columns=["created_at"]
)

print(vacancies_clean.columns.tolist())
print(skills_clean.columns.tolist())

['id', 'title', 'company', 'source_url', 'description_raw', 'experience_years', 'published_at', 'location', 'experience_level', 'title_normalized']
['id', 'name']


In [70]:
# Handle location

vacancies_clean["location"] = (
    vacancies_clean["location"]
    .fillna("Unknown")
)

vacancies_clean["location"].isna().sum()

np.int64(0)

In [71]:
# Handle experience level

vacancies_clean["experience_level"] = (
    vacancies_clean["experience_level"]
    .fillna("Unknown")
)

vacancies_clean["experience_level"].isna().sum()

np.int64(0)

## Experience Data Cleaning

The original experience_years column contains both genuine
zero-experience roles and records where experience information
may not have been specified.

The original column will be preserved. Additional analytical
columns will be created instead of blindly replacing values.

In [72]:
# Create experience status

def classify_experience(row):
    years = row["experience_years"]
    level = row["experience_level"]

    if pd.isna(years):
        return "Missing"

    if years == 0:
        if level in ["Entry level", "Internship", "Junior"]:
            return "Entry / Junior"
        else:
            return "Zero / Unspecified"

    if years <= 2:
        return "1-2 Years"

    if years <= 5:
        return "3-5 Years"

    if years <= 10:
        return "6-10 Years"

    return "10+ Years"


vacancies_clean["experience_category"] = (
    vacancies_clean.apply(classify_experience, axis=1)
)

In [73]:
# Standardize Experience Level

experience_mapping = {
    "Entry level": "Entry Level",
    "Internship": "Internship",
    "Junior": "Junior",
    "Associate": "Associate",
    "Middle": "Mid Level",
    "Mid-Senior level": "Mid-Senior",
    "Senior": "Senior",
    "Lead": "Lead",
    "Not specified": "Not Specified",
    "Not Applicable": "Not Applicable",
    "Unknown": "Unknown"
}

vacancies_clean["experience_level_clean"] = (
    vacancies_clean["experience_level"]
    .map(experience_mapping)
    .fillna("Unknown")
)

vacancies_clean["experience_level_clean"].value_counts()

experience_level_clean
Not Specified     947
Mid Level         368
Entry Level       271
Senior            229
Junior            119
Not Applicable     89
Mid-Senior         53
Lead               29
Associate          28
Unknown            16
Internship          5
Name: count, dtype: int64

In [74]:
# Job Title Standardization

vacancies_clean["title_normalized"] = (
    vacancies_clean["title"]
    .astype("string")
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

vacancies_clean[
    ["title", "title_normalized"]
].head(20)

,title,title_normalized
0,Data Analyst,data analyst
1,Data Analyst,data analyst
2,Data Analyst (f/m/d),data analyst (f/m/d)
3,Data Analyst - Fully Remote,data analyst - fully remote
4,Data Analyst | Remote,data analyst | remote
5,Data Analyst | Remote,data analyst | remote
6,Data Analyst (Product),data analyst (product)
7,Data Analyst,data analyst
8,Reporting Analyst | Remote,reporting analyst | remote
9,Data Analyst,data analyst


In [75]:
# Create job categories

def categorize_job(title):
    title = str(title).lower().strip()

    # Internship
    if "intern" in title:
        return "Data Analyst Intern"

    # Senior roles
    if (
        "senior data analyst" in title
        or "sr. data analyst" in title
        or "sr data analyst" in title
    ):
        return "Senior Data Analyst"

    # Junior roles
    if (
        "junior data analyst" in title
        or "data analyst (junior" in title
        or "data analyst - junior" in title
    ):
        return "Junior Data Analyst"

    # Lead roles
    if "lead data analyst" in title:
        return "Lead Data Analyst"

    # Specialized Data Analyst roles
    if "marketing data analyst" in title:
        return "Marketing Data Analyst"

    if "financial data analyst" in title:
        return "Financial Data Analyst"

    if "healthcare data analyst" in title:
        return "Healthcare Data Analyst"

    if "business data analyst" in title:
        return "Business Data Analyst"

    if "product data analyst" in title:
        return "Product Data Analyst"

    if "market data analyst" in title:
        return "Market Data Analyst"

    # Other analyst roles
    if "reporting analyst" in title:
        return "Reporting Analyst"

    if "business analyst" in title:
        return "Business Analyst"

    # General Data Analyst
    if "data analyst" in title:
        return "Data Analyst"

    return "Other Analyst"


vacancies_clean["job_category"] = (
    vacancies_clean["title"].apply(categorize_job)
)

print("Job categories created successfully!")

vacancies_clean["job_category"].value_counts()


Job categories created successfully!


job_category
Data Analyst               1682
Senior Data Analyst         181
Business Data Analyst        65
Junior Data Analyst          54
Data Analyst Intern          47
Marketing Data Analyst       29
Other Analyst                19
Lead Data Analyst            18
Financial Data Analyst       18
Healthcare Data Analyst      15
Product Data Analyst          9
Market Data Analyst           7
Business Analyst              6
Reporting Analyst             4
Name: count, dtype: int64

In [76]:
# Clean location

vacancies_clean["location"] = (
    vacancies_clean["location"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .fillna("Unknown")
)

In [77]:
# Create location type

def location_type(location):
    location = str(location).lower()

    if location in ["anywhere", "remote"]:
        return "Remote"

    if location in ["unknown", "nan"]:
        return "Unknown"

    return "Onsite / Location Specified"


vacancies_clean["location_type"] = (
    vacancies_clean["location"].apply(location_type)
)

vacancies_clean["location_type"].value_counts()

location_type
Onsite / Location Specified    1942
Remote                          187
Unknown                          25
Name: count, dtype: int64

In [78]:
# Clean the skills table

skills_clean["name"] = (
    skills_clean["name"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print(
    "Unique skill IDs:",
    skills_clean["id"].nunique()
)

print(
    "Unique skill names:",
    skills_clean["name"].nunique()
)

print(
    "Duplicate skill names:",
    skills_clean["name"].duplicated().sum()
)

Unique skill IDs: 2033
Unique skill names: 2033
Duplicate skill names: 0


In [79]:
# Create our first combined dataset


vacancy_skill_data = vacancy_skills_clean.merge(
    skills_clean,
    left_on="skill_id",
    right_on="id",
    how="left"
)

vacancy_skill_data.head()

,vacancy_id,skill_id,id,name
0,1,1,1,SQL
1,1,2,2,Agile
2,1,3,3,ERP Systems
3,1,4,4,Issue Tracking
4,1,5,5,Snowflake


In [80]:
# Add job information

vacancy_skill_data = vacancy_skill_data.merge(
    vacancies_clean,
    left_on="vacancy_id",
    right_on="id",
    how="left",
    suffixes=("_skill", "_vacancy")
)

vacancy_skill_data.shape


(13032, 18)

In [81]:
vacancy_skill_data.head()

,vacancy_id,skill_id,id_skill,name,id_vacancy,title,company,source_url,description_raw,experience_years,published_at,location,experience_level,title_normalized,experience_category,experience_level_clean,job_category,location_type
0,1,1,1,SQL,1,Data Analyst,CRG,https://www.linkedin.com/jobs/view/data-analyst-at-crg-4379681843?trk=public_jobs_topcard-title,Data Analyst\n\n\n\n\nDuration: 6-month contract (potential to extend)\n\nLocation: Remote (EST ...,3.0,NaT,Unknown,Unknown,data analyst,3-5 Years,Unknown,Data Analyst,Unknown
1,1,2,2,Agile,1,Data Analyst,CRG,https://www.linkedin.com/jobs/view/data-analyst-at-crg-4379681843?trk=public_jobs_topcard-title,Data Analyst\n\n\n\n\nDuration: 6-month contract (potential to extend)\n\nLocation: Remote (EST ...,3.0,NaT,Unknown,Unknown,data analyst,3-5 Years,Unknown,Data Analyst,Unknown
2,1,3,3,ERP Systems,1,Data Analyst,CRG,https://www.linkedin.com/jobs/view/data-analyst-at-crg-4379681843?trk=public_jobs_topcard-title,Data Analyst\n\n\n\n\nDuration: 6-month contract (potential to extend)\n\nLocation: Remote (EST ...,3.0,NaT,Unknown,Unknown,data analyst,3-5 Years,Unknown,Data Analyst,Unknown
3,1,4,4,Issue Tracking,1,Data Analyst,CRG,https://www.linkedin.com/jobs/view/data-analyst-at-crg-4379681843?trk=public_jobs_topcard-title,Data Analyst\n\n\n\n\nDuration: 6-month contract (potential to extend)\n\nLocation: Remote (EST ...,3.0,NaT,Unknown,Unknown,data analyst,3-5 Years,Unknown,Data Analyst,Unknown
4,1,5,5,Snowflake,1,Data Analyst,CRG,https://www.linkedin.com/jobs/view/data-analyst-at-crg-4379681843?trk=public_jobs_topcard-title,Data Analyst\n\n\n\n\nDuration: 6-month contract (potential to extend)\n\nLocation: Remote (EST ...,3.0,NaT,Unknown,Unknown,data analyst,3-5 Years,Unknown,Data Analyst,Unknown


In [82]:
# First real business analysis

top_skills = (
    vacancy_skill_data["name"]
    .value_counts()
    .head(20)
)

top_skills

name
SQL                     1298
Python                   760
Excel                    692
Tableau                  685
Power BI                 568
R                        395
Data Visualization       248
Data Analysis            207
SAS                      169
Data Modeling            166
Looker                   145
Snowflake                127
Microsoft Excel          121
Statistical Analysis     114
AWS                      111
ETL                      110
PowerPoint               110
Reporting                102
Azure                     91
Oracle                    87
Name: count, dtype: Int64

In [83]:
# skill demand percentage.

total_skill_vacancies = (
    vacancy_skill_data["vacancy_id"].nunique()
)

skill_demand = (
    vacancy_skill_data
    .groupby("name")["vacancy_id"]
    .nunique()
    .reset_index(name="vacancy_count")
)

skill_demand["demand_percentage"] = (
    skill_demand["vacancy_count"]
    / total_skill_vacancies
    * 100
)

skill_demand = skill_demand.sort_values(
    "vacancy_count",
    ascending=False
)

skill_demand.head(20)

,name,vacancy_count,demand_percentage
1593,SQL,1298,69.300587
1450,Python,760,40.576615
798,Excel,692,36.946076
1790,Tableau,685,36.572344
1391,Power BI,568,30.325681
1477,R,395,21.089162
606,Data Visualization,248,13.240790
465,Data Analysis,207,11.051789
1582,SAS,169,9.022958
536,Data Modeling,166,8.862787


In [84]:
vacancies_clean["experience_category"].value_counts()

experience_category
Zero / Unspecified    774
1-2 Years             390
3-5 Years             376
Missing               321
Entry / Junior        237
6-10 Years             53
10+ Years               3
Name: count, dtype: int64

In [85]:
vacancies_clean["job_category"].value_counts()

job_category
Data Analyst               1682
Senior Data Analyst         181
Business Data Analyst        65
Junior Data Analyst          54
Data Analyst Intern          47
Marketing Data Analyst       29
Other Analyst                19
Lead Data Analyst            18
Financial Data Analyst       18
Healthcare Data Analyst      15
Product Data Analyst          9
Market Data Analyst           7
Business Analyst              6
Reporting Analyst             4
Name: count, dtype: int64

In [86]:
skill_demand.head(20)

,name,vacancy_count,demand_percentage
1593,SQL,1298,69.300587
1450,Python,760,40.576615
798,Excel,692,36.946076
1790,Tableau,685,36.572344
1391,Power BI,568,30.325681
1477,R,395,21.089162
606,Data Visualization,248,13.240790
465,Data Analysis,207,11.051789
1582,SAS,169,9.022958
536,Data Modeling,166,8.862787


## Save the cleaned datasets

In [87]:
from pathlib import Path

cleaned_folder = Path("../data/cleaned")
cleaned_folder.mkdir(parents=True, exist_ok=True)

print("Cleaned folder ready:", cleaned_folder)

Cleaned folder ready: ..\data\cleaned


In [88]:
vacancies_clean.to_csv(
    cleaned_folder / "vacancies_cleaned.csv",
    index=False
)

skills_clean.to_csv(
    cleaned_folder / "skills_cleaned.csv",
    index=False
)

vacancy_skills_clean.to_csv(
    cleaned_folder / "vacancy_skills_cleaned.csv",
    index=False
)

print("All cleaned datasets exported successfully!")

All cleaned datasets exported successfully!


In [89]:
import pandas as pd

# Load the cleaned vacancies CSV
vacancies = pd.read_csv(
    "../data/cleaned/vacancies_cleaned.csv"
)

print("Rows:", len(vacancies))
print("Columns:", list(vacancies.columns))

Rows: 2154
Columns: ['id', 'title', 'company', 'source_url', 'description_raw', 'experience_years', 'published_at', 'location', 'experience_level', 'title_normalized', 'experience_category', 'experience_level_clean', 'job_category', 'location_type']


In [92]:
from pathlib import Path

output_file = Path("../SQL/04_insert_vacancies.sql")

def sql_value(value):
    if pd.isna(value):
        return "NULL"

    value = str(value)
    value = value.replace("\\", "\\\\")
    value = value.replace("'", "''")

    return f"'{value}'"

with open(output_file, "w", encoding="utf-8") as f:

    f.write("USE job_market_intelligence;\n\n")

    for _, row in vacancies.iterrows():

        values = [
            sql_value(row["id"]),
            sql_value(row["title"]),
            sql_value(row["company"]),
            sql_value(row["source_url"]),
            sql_value(row["description_raw"]),
            sql_value(row["experience_years"]),
            sql_value(row["published_at"]),
            sql_value(row["location"]),
            sql_value(row["experience_level"]),
            sql_value(row["title_normalized"]),
            sql_value(row["experience_category"]),
            sql_value(row["experience_level_clean"]),
            sql_value(row["job_category"]),
            sql_value(row["location_type"])
        ]

        query = f"""
INSERT INTO vacancies
(id, title, company, source_url, description_raw,
 experience_years, published_at, location, experience_level,
 title_normalized, experience_category, experience_level_clean,
 job_category, location_type)
VALUES ({", ".join(values)});
"""

        f.write(query)

print("SQL file created successfully:", output_file)
print("Number of rows:", len(vacancies))

SQL file created successfully: ..\SQL\04_insert_vacancies.sql
Number of rows: 2154


In [91]:
print(vacancies.columns.tolist())

['id', 'title', 'company', 'source_url', 'description_raw', 'experience_years', 'published_at', 'location', 'experience_level', 'title_normalized', 'experience_category', 'experience_level_clean', 'job_category', 'location_type']


In [93]:
from pathlib import Path

output_file = Path("../SQL/04_insert_vacancies.sql")

print("File exists:", output_file.exists())
print("File path:", output_file.resolve())

if output_file.exists():
    print("File size:", output_file.stat().st_size, "bytes")

File exists: True
File path: C:\Users\DELL\Desktop\Job-Market-Intelligence-2026\sql\04_insert_vacancies.sql
File size: 4646457 bytes


In [94]:
import pandas as pd

skills = pd.read_csv("../data/cleaned/skills_cleaned.csv")

print("Rows:", len(skills))
print("Columns:", skills.columns.tolist())

Rows: 2033
Columns: ['id', 'name']


In [95]:
from pathlib import Path

output_file = Path("../SQL/05_insert_skills.sql")

def sql_value(value):
    if pd.isna(value):
        return "NULL"

    value = str(value)
    value = value.replace("\\", "\\\\")
    value = value.replace("'", "''")

    return f"'{value}'"

with open(output_file, "w", encoding="utf-8") as f:

    f.write("USE job_market_intelligence;\n\n")

    for _, row in skills.iterrows():

        values = [
            sql_value(row["id"]),
            sql_value(row["name"])
        ]

        query = f"""
INSERT INTO skills
(id, name)
VALUES ({", ".join(values)});
"""

        f.write(query)

print("SQL file created:", output_file.resolve())
print("Number of rows:", len(skills))

SQL file created: C:\Users\DELL\Desktop\Job-Market-Intelligence-2026\sql\05_insert_skills.sql
Number of rows: 2033


In [1]:
import pandas as pd

vacancy_skills = pd.read_csv("../data/cleaned/vacancy_skills_cleaned.csv")

print("Rows:", len(vacancy_skills))
print("Columns:", vacancy_skills.columns.tolist())
print(vacancy_skills.head())

Rows: 13032
Columns: ['vacancy_id', 'skill_id']
   vacancy_id  skill_id
0           1         1
1           1         2
2           1         3
3           1         4
4           1         5


In [2]:
from pathlib import Path

output_file = Path("../SQL/06_insert_vacancy_skills.sql")

def sql_value(value):
    if pd.isna(value):
        return "NULL"

    return str(int(value))

with open(output_file, "w", encoding="utf-8") as f:

    f.write("USE job_market_intelligence;\n\n")

    for _, row in vacancy_skills.iterrows():

        values = [
            sql_value(row["vacancy_id"]),
            sql_value(row["skill_id"])
        ]

        query = f"""
INSERT INTO vacancy_skills
(vacancy_id, skill_id)
VALUES ({", ".join(values)});
"""

        f.write(query)

print("SQL file created:", output_file.resolve())
print("Number of rows:", len(vacancy_skills))

SQL file created: C:\Users\DELL\Desktop\Job-Market-Intelligence-2026\sql\06_insert_vacancy_skills.sql
Number of rows: 13032
